# Map detections from saedump

In [ ]:
from collections import defaultdict
from typing import List, NamedTuple

import pandas as pd
import pybase64
import pydeck
from datetime import datetime
from pydeck.types import String
from visionapi.sae_pb2 import SaeMessage
from visionlib.saedump import DumpMeta, Event, message_splitter


class PointSample(NamedTuple):
    timestamp: datetime
    lat: float
    lon: float
    count: int

INPUT_FILE = '2025-08-28_was-pod01_objectdetector.saedump'

samples: List[PointSample] = []

with open(INPUT_FILE, 'r') as f:
    messages = message_splitter(f)

    start_message = next(messages)
    dump_meta = DumpMeta.model_validate_json(start_message)

    for message in messages:
        event = Event.model_validate_json(message)
        proto_bytes = pybase64.standard_b64decode(event.data_b64)

        sae_msg = SaeMessage()
        sae_msg.ParseFromString(proto_bytes)

        counts = defaultdict(lambda: 0)

        for det in sae_msg.detections:
            counts[det.class_id] += 1

        cam_loc = sae_msg.frame.camera_location
        samples.append(PointSample(datetime.fromtimestamp(sae_msg.frame.timestamp_utc_ms / 1000), cam_loc.latitude, cam_loc.longitude, sum(counts.values())))
        
df_samples = pd.DataFrame(samples, columns=['timestamp', 'lat', 'lon', 'count'])

grid_layer = pydeck.Layer(
    'GridLayer',
    df_samples,
    get_position=['lon', 'lat'],
    get_color_weight='count',
    extruded=False,
    cell_size=20,
    color_aggregation=String('MAX'),
    pickable=True,
)
heatmap_layer = pydeck.Layer(
    'HeatmapLayer',
    df_samples,
    get_position=['lon', 'lat'],
    get_weight='count',
    aggregation=String('MAX'),
    radius_pixels=30,
)
view_state = pydeck.data_utils.compute_view(df_samples[['lon', 'lat']].values.tolist())
pydeck.Deck(
    layers=[grid_layer], 
    initial_view_state=view_state,
    height=800,
).to_html(notebook_display=True, iframe_height=800)



In [ ]:
df_samples.describe(percentiles=[0.8,0.9,0.95,0.98, 0.99])
df_samples.nlargest(50, 'count')